# Kaggle Sweep Phase

In [ ]:
!nvidia-smi
import torch

print("PyTorch:", torch.__version__, "GPUs:", torch.cuda.device_count())

In [ ]:
import os
import shutil

REPO_SOURCE = "git" # change to "kaggle" if using an attached dataset
REPO_URL = "<YOUR_REPO_URL>"

if REPO_SOURCE == "git":
    if not os.path.exists("flowerlite"):
        os.system(f"git clone {REPO_URL} flowerlite")
    os.chdir("flowerlite")
else:
    # Example dataset path
    os.system("cp -r /kaggle/input/flowerlite-repo /kaggle/working/flowerlite")
    os.chdir("/kaggle/working/flowerlite")

os.system("pip install -e .")
import flowerlite  # verify import

print("Successfully imported flowerlite!")

import torchvision

torchvision.datasets.CIFAR10(root='/kaggle/working/data', download=True)
torchvision.datasets.CIFAR100(root='/kaggle/working/data', download=True)


In [ ]:
import os
import subprocess
import time


def measure_sec_per_epoch(gpu_id="0", batch_size=256):
    print(f"Measuring timing on GPU {gpu_id}...")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu_id}
    cmd = ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar10", 
           "--optimizer", "sgd", "--lr", "0.1", "--epochs", "1", "--run-id", "timing_run", 
           "--max-steps", "115", "--data-root", "/kaggle/working/data"]
    
    res = subprocess.run(cmd, env=env, capture_output=True, text=True)
    
    elapsed_100_steps = None
    for line in res.stdout.split("\n"):
        if "Timing (Steps 10-110)" in line:
            elapsed_100_steps = float(line.split(":")[1].replace("s", "").replace("-", "").strip())
            break
            
    if elapsed_100_steps is None:
        print(res.stdout)
        raise RuntimeError("Could not measure timing cleanly. See logs above.")
    
    steps_per_epoch = 45000 / batch_size
    sec_per_epoch = (elapsed_100_steps / 100) * steps_per_epoch
    
    print(f"100 steps took {elapsed_100_steps:.2f}s (excl. start-up & validation). Steps/s: {100 / elapsed_100_steps:.2f}")
    return sec_per_epoch

sec_per_epoch = measure_sec_per_epoch("0")
print(f"Measured sec/epoch: {sec_per_epoch:.2f}")

planned_epochs = 4 * 30 # 4 configs * 30 epochs
total_hours = (sec_per_epoch * planned_epochs) / 3600
print(f'Projected wall-clock: {total_hours:.2f} hours')

if total_hours > 6.0:
    raise Exception("Projection exceeds 6 hours. Consider lowering epoch count. Halting.")

print('Starting Sweep in background...')
cmds_0 = [
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar10", "--optimizer", "sgd", "--lr", "0.10", "--epochs", "30", "--run-id", "sweep_c10_sgd_10", "--data-root", "/kaggle/working/data"],
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar10", "--optimizer", "sgd", "--lr", "0.15", "--epochs", "30", "--run-id", "sweep_c10_sgd_15", "--data-root", "/kaggle/working/data"],
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar10", "--optimizer", "adam", "--lr", "0.001", "--epochs", "30", "--run-id", "sweep_c10_adam_001", "--data-root", "/kaggle/working/data"],
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar10", "--optimizer", "adam", "--lr", "0.002", "--epochs", "30", "--run-id", "sweep_c10_adam_002", "--data-root", "/kaggle/working/data"]
]

cmds_1 = [
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar100", "--optimizer", "sgd", "--lr", "0.10", "--epochs", "30", "--run-id", "sweep_c100_sgd_10", "--data-root", "/kaggle/working/data"],
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar100", "--optimizer", "sgd", "--lr", "0.15", "--epochs", "30", "--run-id", "sweep_c100_sgd_15", "--data-root", "/kaggle/working/data"],
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar100", "--optimizer", "adam", "--lr", "0.001", "--epochs", "30", "--run-id", "sweep_c100_adam_001", "--data-root", "/kaggle/working/data"],
    ["python", "train.py", "--config", "configs/base.yaml", "--dataset", "cifar100", "--optimizer", "adam", "--lr", "0.002", "--epochs", "30", "--run-id", "sweep_c100_adam_002", "--data-root", "/kaggle/working/data"]
]

def run_sequence(cmds, gpu, log_file):
    with open(log_file, "w") as f:
        f.write(f"--- Starting GPU {gpu} sequence ---\n")
    
    script_path = f"/kaggle/working/run_gpu{gpu}.sh"
    with open(script_path, "w") as f:
        f.write("#!/bin/bash\n")
        for cmd in cmds:
            f.write(" ".join(cmd) + f" >> {log_file} 2>&1\n")
            f.write(f"cp -r experiments/{cmd[13]} /kaggle/working/\n") # copy run dir out
    
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu}
    return subprocess.Popen(["bash", script_path], env=env)

p0 = run_sequence(cmds_0, "0", "/kaggle/working/log_gpu0.txt")
p1 = run_sequence(cmds_1, "1", "/kaggle/working/log_gpu1.txt")

import sys

while p0.poll() is None or p1.poll() is None:
    time.sleep(30)
    print("--- GPU 0 Last 15 Lines ---")
    os.system("tail -n 15 /kaggle/working/log_gpu0.txt")
    print("--- GPU 1 Last 15 Lines ---")
    os.system("tail -n 15 /kaggle/working/log_gpu1.txt")
    sys.stdout.flush()

p0.wait()
p1.wait()

print("Sweep Complete. Output directories copied to /kaggle/working/")
# Compile summary json
import json

with open("/kaggle/working/summary.json", "w") as f:
    json.dump({"status": "sweep_completed"}, f)
